# Task 33
We are asked to compute the exceedance probability of a light tailed distribution. Here we use the normal(1,1). We are interested in computing
$$P(S_n > x)$$
where $S_n = X_1 + ... + X_n$ and $X_i \sim N(1,1)$. Now note that

$$ \bar{X} = \frac{S_n}{n} \implies S_n = n \bar{X}$$
As such we can equivalently compute
$$P(\bar{X} > \frac{x}{n})$$
This rewriting allows us to use 6.2 "Exponential tilting for sums". Further it is useful to note that the expectation of a RV and the expectation of the the sample mean are equivalent due to the linearity of expectation. So
$$
E_\theta[\bar{X}] = E_\theta[X_i]
$$
This means that we can just tilt the underlying distribution of the $X_i$ and compute the sample mean of the tilted distribution to estimate $P(\bar{X} > \frac{x}{n}) = P(S_n > x)$.

Now the cumulant generating function of a normal distribution is
$$
\kappa(\theta) = \mu \theta + \frac{\sigma^2 \theta^2}{2}
$$
So specifically for $N(1,1)$ we have
$$
\kappa(\theta) = \theta + \frac{\theta^2}{2}
$$
We can tilt $f(x)$ by $e^{\theta x}$ to get a new distribution $f_\theta = e^{\theta x - \kappa(\theta)}$ which has the general form
$$N_\theta(\mu + \sigma^2 \theta, \sigma^2)$$
Now we want to tilt the distribution such that the mean of the tilted distribution is equal to $\frac{x}{n}$. So we want to solve
$$
\mu + \sigma^2 \theta = \frac{x}{n} \implies \theta = \dfrac{\frac{x}{n} - \mu}{\sigma^2}
$$
Which for $n=10$, $\mu = 1$, $\sigma^2 = 1$, $x_1, x_2 = 20,25$ gives us
$$\theta_1 = 1,\quad \theta_2 = 1.5 $$

In [179]:
import numpy as np

N = 10000
n = 10
sigma = 1
mu = 1
x = [20, 25]
theta = [(x_i/n - mu) / sigma**2 for x_i in x]

# Sample from the tilted distributions (3D array)
X = np.array([np.random.normal(mu + sigma**2 * t, sigma, size=(n, N)) for t in theta])
# Compress this to means before we normalize by the inverse tilt
X_bar = np.mean(X, axis=1)

# Cumulant generating function
CGF = lambda theta: mu * theta + (sigma**2 * theta**2) / 2

def weights_sum(theta, X_bar, n, CGF):
    """Weights for change of measure for X_bar ->
    Inverse of the exponential tilt"""
    w = np.exp(-n * (theta * X_bar - CGF(theta)))
    return w

def compute_exceedance(theta, X_bar, n, CGF, x):
    """ Estimate E_t[1(X_bar>x)] under tilted measure """

    weights = weights_sum(theta, X_bar, n, CGF)
    ell = X_bar > x
    ell = ell*weights
    # Empircal variance
    emp_var = np.var(ell) / N
    # Estimate
    ell = np.mean(ell)

    return ell, emp_var

for i in range(len(x)):
    ex_prob, emp_var = compute_exceedance(theta[i], X_bar[i, :], n, CGF, x[i] / n)
    print(f"Exceedance probability for P(X_bar >{x[i] / n})", ex_prob)
    print(f"Empirical variance for estimator", emp_var)

Exceedance probability for P(X_bar >2.0) 0.0007514765672949321
Empirical variance for estimator 2.078654175596842e-10
Exceedance probability for P(X_bar >2.5) 1.0721490310108646e-06
Empirical variance for estimator 6.033928934376314e-16


We can compare this with the theoretical value of the exceedance probability. Note that
$$
\bar{X} \sim N(\mu, \sigma^2/n)
$$
So we can compute the exceedance probability as
$$
P(\bar{X} > x/n) = 1 - \Phi\left(\frac{x/n - \mu}{\sigma/\sqrt{n}}\right)
$$

In [180]:
from scipy.stats import norm
SF = lambda x: norm(loc=1, scale=np.sqrt(1 / n)).sf(x)
SF(2)

np.float64(0.000782701129001274)

In [181]:
SF(2.5)

np.float64(1.0507179780062186e-06)

We can also use the refined estimator from Asmussen-Kroese to estimate $P(S_n > x)$ by using the result from exercise 28, so we can write

$$
P(S_n > x) = n P(S_n > x, M_n = X_n) = n E[\bar F(max(M_{n-1}, x - S_{n-1}))]
$$
Note now we should not sample from the tilted distribution, but rather the original distribution.

In [193]:
N = 100*10000
X = np.random.normal(mu, sigma, size=(2, n, N))

# Maximum of the first n-1 RV's for all N samples
M = np.max(X[:,:n-1, :], axis = 1)
# Sum of the first n-1 RV
S = np.sum(X[:,:n-1,:], axis=1)
x = np.array(x)

Y = np.array([np.maximum(M[i,:], x[i] - S[i,:]) for i in range(X.shape[0])])

ell = n * norm(loc=1, scale=1).sf(Y)

vari = np.var(ell, axis = 1) / N
print(vari)
ell = np.mean(ell, axis = 1)
print(ell)


[1.31465829e-10 3.66004348e-14]
[7.88743145e-04 1.08762272e-06]
